# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import torch
from typing import List, Tuple, Union
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import *
from scipy.stats import boxcox, chi2
from statistics import NormalDist
from sklearn.linear_model import LinearRegression
from sklearn.covariance import EmpiricalCovariance
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly import express as px
import plotly.figure_factory as ff
from sklearn.preprocessing import LabelEncoder
import shap
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from lightgbm import LGBMRegressor
import lightgbm as lgb
import optuna
import warnings
import random
warnings.simplefilter('ignore')

In [ ]:
class Plots:
    def __init__(self):
        pass

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return data
    
    def check_2d_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        return np.array(data)
    
    def histogram_and_box_plot(self, data, name="", with_annotation=True):
        data = self.check_data(data=data)
        fig = make_subplots(rows=1, cols=2, specs=[[{"type": "histogram"}, {"type": "box"}]])
        fig.add_trace(go.Box(y=data, name='', marker=dict(color="rgb(48,70,116)"), showlegend=False), row=1, col=1)
        fig.add_trace(go.Histogram(x=data, marker=dict(color="rgb(48,70,116)"), showlegend=False), row=1, col=2)
        if(with_annotation==True):
            for x in zip(["Min","Q1","Med","Q3","Max"], np.quantile(data, [0, 0.25, 0.5, 0.75, 1])):
                fig.add_annotation(x=0.4, y=x[1], text=x[0] + ": " + str(np.round(x[1], 3)), showarrow=False)
        fig.update_layout(template="simple_white", width=1600, height=800, title=f"<b>{name.title()} distribution<b>", title_x=0.5, font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")
    
    def pie_and_bar_plot(self, data, name=""):
        data = self.check_data(data=data)
        data = list(data)
        rows = 1
        columns = 2
        labels, frequency = np.unique(data, return_counts=True)
        sorted_indices = np.argsort(frequency)[::-1]
        labels = labels[sorted_indices]
        frequency = frequency[sorted_indices]
        colors = px.colors.qualitative.Dark24
        fig = make_subplots(rows=rows, cols=columns, specs=[[{"type": "pie"}, {"type": "bar"}] for i in range(0, rows)])
        fig.add_trace(go.Pie(values=frequency, labels=labels, showlegend=True, textinfo='value+percent', hole=0.3, marker=dict(line=dict(color='black', width=2), colors=colors)), row=1, col=1)
        fig.add_trace(go.Bar(x=labels, y=frequency, marker=dict(line=dict(color='black', width=1), color=colors), showlegend=False), row=1, col=2)
        fig.update_layout(template="simple_white", width=1600, height=800, title=f"<b>Pie chart and bar chart {name.title()}<b>", title_x=0.5, font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")

    def box_and_hist_plot(self, train, test, name="", bin_size=1):
        train = self.check_data(data=train)
        test = self.check_data(data=test)
        rows = 1
        columns = 2
        fig = make_subplots(rows=rows, cols=columns, specs=[[{"type": "box"}, {"type": "histogram"}] for i in range(0, rows)])
        fig.add_trace(go.Box(y=train, showlegend=True, name="Train", marker=dict(color='blue')), row=1, col=1)
        fig.add_trace(go.Box(y=test, showlegend=True, name="Test", marker=dict(color='red')), row=1, col=1)
        fig.add_trace(go.Histogram(x=train, showlegend=False, name="Train", marker=dict(color='blue')), row=1, col=2)
        fig.add_trace(go.Histogram(x=test, showlegend=False, name="Test", marker=dict(color='red'), xbins=dict(size=bin_size)), row=1, col=2)
        fig.update_layout(template="simple_white", width=1600, height=800, title=f"<b>Box plot and histogram {name.title()}<b>", title_x=0.5, font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")

    def boxplot_by_categorical(self, data, categorical, target, name=""):
        fig = go.Figure()
        labels, frequency = np.array(data[categorical].value_counts().index), np.array(data[categorical].value_counts().values)
        sorted_indices = np.argsort(frequency)[::-1]
        labels = labels[sorted_indices]
        frequency = frequency[sorted_indices]
        colors = px.colors.qualitative.Dark24
        for color_idx, category in enumerate(labels):
            indices = list(np.where(data[categorical]==category)[0])
            grouped_data = list(data[target][indices])
            try:
                fig.add_trace(go.Box(y=grouped_data, name=str(category), marker=dict(color=colors[color_idx]), showlegend=True))
            except:
                fig.add_trace(go.Box(y=grouped_data, name=str(category), marker=dict(color=colors[random.randint(0, len(colors)-1)]), showlegend=True))
        fig.update_layout(template="simple_white", width=1600, height=800, title=f"<b>Box plot of {target} for each category of {categorical}<b>", title_x=0.5, font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")

    def linear_regression_plot(self, data, feature, target):
        model = LinearRegression()
        no_nan_indices = ~data[feature].isna()
        model.fit(data[no_nan_indices][feature].values.reshape(-1, 1), data[no_nan_indices][target])
        predictions = model.predict(data[no_nan_indices][feature].values.reshape(-1, 1))
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=data[no_nan_indices][feature], y=data[no_nan_indices][target], mode='markers', name='Real values'))
        fig.add_trace(go.Scatter(x=data[no_nan_indices][feature], y=predictions, mode='lines', line_color="green", name='Expected values'))
        confidence_interval = 0.95
        def confidence_interval(data, confidence=0.95):
            dist = NormalDist.from_samples(data)
            z = NormalDist().inv_cdf((1 + confidence) / 2.)
            h = dist.stdev * z / ((len(data) - 1) ** .5)
            return h
        h = confidence_interval(data[no_nan_indices][target])
        fig.add_traces(go.Scatter(x=data[no_nan_indices][feature], y=predictions + h, mode='lines', line_color = 'rgba(0,0,0,0)', showlegend=False))
        fig.add_trace(go.Scatter(x=data[no_nan_indices][feature], y=predictions - h, mode='lines', name='Confidence interval', line_color = 'rgba(0,0,0,0)', showlegend=True, fill='tonexty', fillcolor = 'rgba(0, 255, 0, 0.2)'))
        fig.update_layout(template="simple_white", width=1600, height=800, title_text=f"<b>Linear regression between {feature} and {target}<b>", title_x=0.5, xaxis_title=feature, yaxis_title=target, font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")
    
    def barplot_missing_values(self, data, features_names, name=""):
        data = self.check_data(data=data)
        fig = go.Figure()
        fig.add_trace(go.Bar(x=features_names, y=data, marker=dict(color=data, colorscale='viridis', line=dict(color='black', width=1))))
        fig.update_layout(template="simple_white", width=max(30*len(features_names), 600), height=max(30*len(features_names), 600), title=f"<b>Bar chart {name.title()}<b>", title_x=0.5, yaxis_title="Frequency", xaxis=dict(title="Features", showticklabels=True), font=dict(family="Times New Roman", size=16 ,color="Black"))
        fig.show("png")
    
    def tsne_plot(self, tsne_data, outliers):
        non_outliers = np.setdiff1d(np.arange(tsne_data.shape[0]), outliers)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=tsne_data[non_outliers, 0], y=tsne_data[non_outliers, 1], mode='markers', marker=dict(color="green", size=7, line=dict(color='black', width=1)), name=f"Non-Outliers", showlegend=True))
        fig.add_trace(go.Scatter(x=tsne_data[outliers, 0], y=tsne_data[outliers, 1], mode='markers', marker=dict(color="red", size=10, line=dict(color='black', width=1)), name=f"Outliers", showlegend=True))
        fig.update_layout(template="simple_white", width=800, height=800, title="<b>TSNE plot<b>", title_x=0.5, font=dict(family="Times New Roman",size=16,color="Black"), legend_itemsizing ='trace')
        fig.show("png")
    
    def plot_feature_importances(self, results, importance_type):
        sorted = results.sort_values(by=f'{importance_type}_mean', ascending=False)
        colors = px.colors.qualitative.Plotly
        fig = go.Figure()
        fig.add_trace(go.Bar(x=sorted['feature'], y=sorted[f'{importance_type}_mean'], marker=dict(color=sorted[f'{importance_type}_mean'], colorscale=colors, line=dict(color='black', width=1)), error_y=dict(type='data', array=sorted[f'{importance_type}_std'], visible=True)))
        fig.update_layout(template="simple_white", width=1600, height=800, title_text=f"<b>Feature importance {importance_type.title()}<b>", title_x=0.5, yaxis_title="Feature importance", xaxis=dict(title='Features', showticklabels=True, type="category"), font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")
    
    def simple_feature_importance(self, feature_importance, importance_type):
        sorted = feature_importance.sort_values(by=importance_type, ascending=False)
        colors = px.colors.qualitative.Plotly
        fig = go.Figure()
        fig.add_trace(go.Bar(x=sorted.index, y=sorted[importance_type], marker=dict(color=sorted[importance_type], colorscale=colors, line=dict(color='black', width=1))))
        fig.update_layout(template="simple_white", width=1600, height=800, title_text="<b>Feature importance<b>", title_x=0.5, yaxis_title="Feature importance", xaxis=dict(title='Features', showticklabels=True, type="category"), font=dict(family="Times New Roman",size=16,color="Black"))
        fig.show("png")

    def compare_predictions_with_real_values(self, y_true, y_pred, metric="MSE"):
        self.metric = metric
        metrics = { "MSE": self.mean_squared_error(y_true, y_pred),
                    "RMSE": self.root_mean_squared_error(y_true, y_pred),
                    "MAE": self.mean_absolute_error(y_true, y_pred),
                    "MAPE": self.mean_absolute_percentage_error(y_true, y_pred),
                    "MedAE": self.median_absolute_error(y_true, y_pred),
                    "MSLE": self.mean_squared_logarithm_error(y_true, y_pred)}
        if self.metric not in metrics:
            raise ValueError('Unsupported metric: {}'.format(metric))
        self.eval_metric = np.round(metrics[self.metric], 5)
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[i for i in range(len(y_true))], y=y_true.flatten().tolist(), mode='lines', line=dict(color="orange"), name="Real values"))
        fig.add_trace(go.Scatter(x=[i for i in range(len(y_true))], y=y_pred.flatten().tolist(), mode='lines', line=dict(color="blue"), name="Predictions"))
        fig.update_layout(template="simple_white", width=600, height=600, title="<b>Predictions and Real values<b>", title_x=0.5, xaxis_title="Observation", yaxis_title="Values", font=dict(family="Times New Roman",size=16,color="Black"), legend_title_text='{}: {}'.format(self.metric.upper(), self.eval_metric))
        fig.show("png")
    def mean_squared_error(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)
    def root_mean_squared_error(self, y_true, y_pred):
        return np.sqrt(np.mean((y_true - y_pred) ** 2))
    def mean_absolute_error(self, y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))
    def mean_absolute_percentage_error(self, y_true, y_pred):
        return np.mean(np.abs((y_true - y_pred) / y_true))
    def median_absolute_error(self, y_true, y_pred):
        return np.median(np.abs(y_true - y_pred))
    def mean_squared_logarithm_error(self, y_true, y_pred):
        return np.mean((np.log(np.abs(1+y_true))-np.log(np.abs(1+y_pred)))**2)
    
plots = Plots()

# Problem understanding

$\text{Predict the sales price for each house.}$

# Data loading

In [ ]:
train_data = pd.read_csv("input/train.csv")
target_variable = "SalePrice"
train_data.head()

In [ ]:
test_data = pd.read_csv("input/test.csv")
test_data.head()

# Exploratory Data Analysis

<a id="6"></a>
## Base information about data

In [ ]:
def base_information(data):
    df =  pd.DataFrame(data.dtypes, columns=['dtypes'])
    df['Number of missing values'] = data.isna().sum()
    df['Percentage of missing values'] = data.isna().sum()/data.shape[0]
    df['Unique values'] = data.nunique().values
    df['Count'] = data.count().values
    return df.style.background_gradient(cmap='Blues')
base_information(train_data)

In [ ]:
base_information(test_data)

$\text{Some of our features contains missing values. We will take care of them in next section.}$<p>
$\text{We can also see that there are some object and continous features.}$

$\text{We wil also define lists with one\_hot, ordinal features and continous features}$

In [ ]:
one_hot, ordinal, continous_features = [], [], []

## Features exploration

$\text{Firstly, let's find out whether Id contains unique values.}$

In [ ]:
print("Id are unique values (1 per row): {}".format(len(train_data["Id"].value_counts()) == len(train_data)))

In [ ]:
features_to_drop = []
features_to_drop.append("id")
train_data.drop("Id", axis=1, inplace=True)
test_data.drop("Id", axis=1, inplace=True)

### Target variable

In [ ]:
plots.histogram_and_box_plot(data=train_data[target_variable], name=target_variable, with_annotation=True)

$\text{Target variable - SalePrice has right skewed distribution.}$<p>
$\text{We will use log transformation in the next stage to make it more normally distributed.}$

### Categorical features

#### MSSubClass

$\text{Although MSSubClass is a numerical feature, it is actually a categorical feature.}$<p>
$\text{We can see that there are 15 unique values.}$

In [ ]:
train_data["MSSubClass"] = train_data["MSSubClass"].astype(str)
test_data["MSSubClass"] = test_data["MSSubClass"].astype(str)
train_data["MSSubClass"] = train_data["MSSubClass"].map({'20': '1-STORY 1946 & NEWER ALL STYLES', '30': '1-STORY 1945 & OLDER', '40': '1-STORY W/FINISHED ATTIC ALL AGES', '45': '1-1/2 STORY - UNFINISHED ALL AGES', '50': '1-1/2 STORY FINISHED ALL AGES', '60': '2-STORY 1946 & NEWER', '70': '2-STORY 1945 & OLDER', '75': '2-1/2 STORY ALL AGES', '80': 'SPLIT OR MULTI-LEVEL', '85': 'SPLIT FOYER', '90': 'DUPLEX - ALL STYLES AND AGES', '120': '1-STORY PUD (Planned Unit Development) - 1946 & NEWER', '150': '1-1/2 STORY PUD - ALL AGES', '160': '2-STORY PUD - 1946 & NEWER', '180': 'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER', '190': '2 FAMILY CONVERSION - ALL STYLES AND AGES'})
test_data["MSSubClass"] = test_data["MSSubClass"].map({'20': '1-STORY 1946 & NEWER ALL STYLES', '30': '1-STORY 1945 & OLDER', '40': '1-STORY W/FINISHED ATTIC ALL AGES', '45': '1-1/2 STORY - UNFINISHED ALL AGES', '50': '1-1/2 STORY FINISHED ALL AGES', '60': '2-STORY 1946 & NEWER', '70': '2-STORY 1945 & OLDER', '75': '2-1/2 STORY ALL AGES', '80': 'SPLIT OR MULTI-LEVEL', '85': 'SPLIT FOYER', '90': 'DUPLEX - ALL STYLES AND AGES', '120': '1-STORY PUD (Planned Unit Development) - 1946 & NEWER', '150': '1-1/2 STORY PUD - ALL AGES', '160': '2-STORY PUD - 1946 & NEWER', '180': 'PUD - MULTILEVEL - INCL SPLIT LEV/FOYER', '190': '2 FAMILY CONVERSION - ALL STYLES AND AGES'})
plots.pie_and_bar_plot(data=train_data["MSSubClass"], name="MSSubClass")

In [ ]:
plots.pie_and_bar_plot(data=test_data["MSSubClass"], name="MSSubClass")

$\text{Most of the houses are 1-story 1946 and newer.}$<p>
$\text{There are also some houses with 2-story 1946 and newer.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("MSSubClass")

#### MSZoning

In [ ]:
plots.pie_and_bar_plot(data=train_data["MSZoning"], name="MSZoning (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["MSZoning"], name="MSZoning (test data)")

$\text{We have missing value in the test set.}$<p>
$\text{Based on the data description, it might be: A - Agriculture, I - Industrial or  RP - Residential Low Density Park.}$<p>
$\text{We will encode it with "Other" category.}$

In [ ]:
test_data["MSZoning"].fillna("Other", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("MSZoning")

#### Street

In [ ]:
plots.pie_and_bar_plot(data=train_data["Street"], name="Street (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Street"], name="Street (test data)")

$\text{Most of the houses have Paved street.}$<p>
$\text{Only 6 houses have Gravel street.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Street")

#### Alley

In [ ]:
plots.pie_and_bar_plot(data=train_data["Alley"], name="Alley (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Alley"], name="Alley (test data)")

$\text{There are a lot of missing values in Alley feature.}$<p>
$\text{It probably means that there is no alley access.}$<p>
$\text{We will fill missing values with the test data category - No alley access.}$

In [ ]:
train_data["Alley"].fillna("No alley access", inplace=True)
test_data["Alley"].fillna("No alley access", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Alley")

#### LotShape

In [ ]:
plots.pie_and_bar_plot(data=train_data["LotShape"], name="LotShape (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["LotShape"], name="LotShape (test data)")

$\text{Most of the houses have Regular shape.}$<p>
$\text{There are not many houses with Irregular shape and the most frequent one is IR1.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("LotShape")

#### LandContour

In [ ]:
plots.pie_and_bar_plot(data=train_data["LandContour"], name="LandContour (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["LandContour"], name="LandContour (test data)")

$\text{Most of the houses have Lvl contour.}$<p>
$\text{There are not many houses with other contours.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("LandContour")

#### Utilities

In [ ]:
plots.pie_and_bar_plot(data=train_data["Utilities"], name="Utilities (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Utilities"], name="Utilities (test data)")

$\text{The data is highly imbalanced.}$<p>
$\text{In test data there is nan value, so we will fill it with the "Other" category.}$

In [ ]:
test_data["Utilities"].fillna("Other", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Utilities")

#### LotConfig

In [ ]:
plots.pie_and_bar_plot(data=train_data["LotConfig"], name="LotConfig (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["LotConfig"], name="LotConfig (test data)")

$\text{Most of the houses are inside lot configuration.}$<p>
$\text{There are not many houses with other configurations.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("LotConfig")

#### LandSlope

In [ ]:
plots.pie_and_bar_plot(data=train_data["LandSlope"], name="LandSlope (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["LandSlope"], name="LandSlope (test data)")

$\text{Most of the houses have gentle slope.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("LandSlope")

#### Neighborhood

In [ ]:
plots.pie_and_bar_plot(data=train_data["Neighborhood"], name="Neighborhood (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Neighborhood"], name="Neighborhood (test data)")

$\text{There are a lot of different types of neighborhoods.}$<p>
$\text{Most of the houses are in NAmes neighborhood.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Neighborhood")

#### Condition1

In [ ]:
plots.pie_and_bar_plot(data=train_data["Condition1"], name="Condition1 (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Condition1"], name="Condition1 (test data)")

$\text{Most of the houses are in Norm condition.}$<p>
$\text{There are not many houses with other conditions.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Condition1")

#### Condition2

In [ ]:
plots.pie_and_bar_plot(data=train_data["Condition2"], name="Condition2 (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Condition2"], name="Condition2 (test data)")

$\text{Interesting fact is that there are some categories in train data that are not in test data.}$<p>
$\text{We can replace all categories except of most frequent one with the category - Other.}$<p>
$\text{We will take care of this in the next section.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Condition2")

#### BldgType

In [ ]:
plots.pie_and_bar_plot(data=train_data["BldgType"], name="BldgType (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BldgType"], name="BldgType (test data)")

$\text{The distribution of BldgType is simmilar in train and test data.}$<p>
$\text{Most of the houses are 1Fam type.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("BldgType")

#### HouseStyle

In [ ]:
plots.pie_and_bar_plot(data=train_data["HouseStyle"], name="HouseStyle (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["HouseStyle"], name="HouseStyle (test data)")

$\text{The 2.5Fin category is not in test data.}$<p>
$\text{That is actually not a huge problem, because after encoding we will have one less category in the test data.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("HouseStyle")

#### RoofStyle

In [ ]:
plots.pie_and_bar_plot(data=train_data["RoofStyle"], name="RoofStyle (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["RoofStyle"], name="RoofStyle (test data)")

$\text{Most of the houses have Gable roof style, the second most frequent is Hip.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("RoofStyle")

#### RoofMatl

In [ ]:
plots.pie_and_bar_plot(data=train_data["RoofMatl"], name="RoofMatl (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["RoofMatl"], name="RoofMatl (test data)")

$\text{The data is highly imbalanced.}$<p>
$\text{We could probably replace all categories except of CompShg with the category - Other.}$<p>
$\text{We will take care of this in the next section.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("RoofMatl")

#### Exterior1st

In [ ]:
plots.pie_and_bar_plot(data=train_data["Exterior1st"], name="Exterior1st (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Exterior1st"], name="Exterior1st (test data)")

$\text{There are missing values in test data.}$<p>
$\text{We will fill them with the Other category.}$

In [ ]:
test_data["Exterior1st"].fillna("Other", inplace=True)

In [ ]:
for category in train_data["Exterior1st"].value_counts().index:
    if category not in test_data["Exterior1st"].value_counts().index:
        print(category)

$\text{Also we will replace all rows that contain Stone and ImStucc in train data with the category - Other, beacause they are not present in test data.}$

In [ ]:
test_data["Exterior1st"].fillna("Other", inplace=True)
train_data["Exterior1st"] = train_data["Exterior1st"].replace("Stone", "Other")
train_data["Exterior1st"] = train_data["Exterior1st"].replace("ImStucc", "Other")

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Exterior1st")

#### Exterior2nd

In [ ]:
plots.pie_and_bar_plot(data=train_data["Exterior2nd"], name="Exterior2nd (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Exterior2nd"], name="Exterior2nd (test data)")

$\text{Simmilarly to Exterior1st, we will fill missing values with the Other category.}$

In [ ]:
test_data["Exterior2nd"].fillna("Other", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Exterior2nd")

#### MasVnrType

In [ ]:
plots.pie_and_bar_plot(data=train_data["MasVnrType"], name="MasVnrType (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["MasVnrType"], name="MasVnrType (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - None.}$

In [ ]:
train_data["MasVnrType"].fillna("None", inplace=True)
test_data["MasVnrType"].fillna("None", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("MasVnrType")

#### ExterQual

In [ ]:
plots.pie_and_bar_plot(data=train_data["ExterQual"], name="ExterQual (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["ExterQual"], name="ExterQual (test data)")

$\text{Most of the houses have average quality of the material on the exterior.}$<p>
$\text{There are no houses with poor quality.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("ExterQual")

#### ExterCond

In [ ]:
plots.pie_and_bar_plot(data=train_data["ExterCond"], name="ExterCond (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["ExterCond"], name="ExterCond (test data)")

$\text{Most of the houses have average condition of the material on the exterior.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("ExterCond")

#### Foundation

In [ ]:
plots.pie_and_bar_plot(data=train_data["Foundation"], name="Foundation (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Foundation"], name="Foundation (test data)")

$\text{The data is very simmilar in train and test data.}$<p>
$\text{Most of the houses have PConc foundation.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Foundation")

#### BsmtQual

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtQual"], name="BsmtQual (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtQual"], name="BsmtQual (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - No basement.}$

In [ ]:
train_data["BsmtQual"].fillna("No basement", inplace=True)
test_data["BsmtQual"].fillna("No basement", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtQual")

#### BsmtCond

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtCond"], name="BsmtCond (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtCond"], name="BsmtCond (test data)")

$\text{Simmilarly to BsmtQual, we will fill missing values with the category - No basement.}$

In [ ]:
train_data["BsmtCond"].fillna("No basement", inplace=True)
test_data["BsmtCond"].fillna("No basement", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtCond")

#### BsmtExposure

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtExposure"], name="BsmtExposure (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtExposure"], name="BsmtExposure (test data)")

$\text{Simmilarly to BsmtQual, we will fill missing values with the category - No basement.}$

In [ ]:
train_data["BsmtExposure"].fillna("No basement", inplace=True)
test_data["BsmtExposure"].fillna("No basement", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtExposure")

#### BsmtFinType1

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtFinType1"], name="BsmtFinType1 (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtFinType1"], name="BsmtFinType1 (test data)")

$\text{Simmilarly to BsmtQual, we will fill missing values with the category - No basement.}$

In [ ]:
train_data["BsmtFinType1"].fillna("No basement", inplace=True)
test_data["BsmtFinType1"].fillna("No basement", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtFinType1")

#### BsmtFinType2

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtFinType2"], name="BsmtFinType2 (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtFinType2"], name="BsmtFinType2 (test data)")

$\text{Simmilarly to BsmtQual, we will fill missing values with the category - No basement.}$

In [ ]:
train_data["BsmtFinType2"].fillna("No basement", inplace=True)
test_data["BsmtFinType2"].fillna("No basement", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtFinType2")

#### Heating

In [ ]:
plots.pie_and_bar_plot(data=train_data["Heating"], name="Heating (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Heating"], name="Heating (test data)")

$\text{Most of the houses have GasA heating.}$<p>
$\text{There are not many houses with other types of heating.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Heating")

#### HeatingQC

In [ ]:
plots.pie_and_bar_plot(data=train_data["HeatingQC"], name="HeatingQC (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["HeatingQC"], name="HeatingQC (test data)")

$\text{Most of the houses have good quality and condition of heating.}$<p>
$\text{There are not many houses with poor quality and condition of heating.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("HeatingQC")

#### CentralAir

In [ ]:
plots.pie_and_bar_plot(data=train_data["CentralAir"], name="CentralAir (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["CentralAir"], name="CentralAir (test data)")

$\text{It is binary feature.}$<p>
$\text{Most of the houses have central air conditioning.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("CentralAir")

#### Electrical

In [ ]:
plots.pie_and_bar_plot(data=train_data["Electrical"], name="Electrical (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Electrical"], name="Electrical (test data)")

$\text{There is one missing value in test data.}$<p>
$\text{We will fill it in the next section.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Electrical")

#### BsmtFullBath

$\text{Although BsmtFullBath is a numerical feature, it is actually a categorical feature.}$<p>
$\text{We can see that there are 4 unique values.}$

In [ ]:
train_data["BsmtFullBath"] = train_data["BsmtFullBath"].astype(str)
test_data["BsmtFullBath"] = test_data["BsmtFullBath"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtFullBath"], name="BsmtFullBath (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtFullBath"], name="BsmtFullBath (test data)")

$\text{There are some missing values in test data.}$<p>
$\text{We will take care of them in the next section.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtFullBath")

#### BsmtHalfBath

$\text{Although BsmtHalfBath is a numerical feature, it is actually a categorical feature.}$

In [ ]:
train_data["BsmtHalfBath"] = train_data["BsmtHalfBath"].astype(str)
test_data["BsmtHalfBath"] = test_data["BsmtHalfBath"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["BsmtHalfBath"], name="BsmtHalfBath (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BsmtHalfBath"], name="BsmtHalfBath (test data)")

$\text{There are some missing values in test data.}$<p>
$\text{We will take care of them in the next section.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BsmtHalfBath")

#### FullBath

$\text{Although FullBath is a numerical feature, it is actually a categorical feature.}$

In [ ]:
train_data["FullBath"] = train_data["FullBath"].astype(str)
test_data["FullBath"] = test_data["FullBath"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["FullBath"], name="FullBath (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["FullBath"], name="FullBath (test data)")

$\text{There is extra category in test data.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("FullBath")

#### HalfBath

$\text{Although BsmtFullBath is a numerical feature, it is actually a categorical feature.}$

In [ ]:
train_data["HalfBath"] = train_data["HalfBath"].astype(str)
test_data["HalfBath"] = test_data["HalfBath"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["HalfBath"], name="HalfBath (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["HalfBath"], name="HalfBath (test data)")

$\text{Most of the houses have no half baths.}$<p>
$\text{There are not many houses with 2 half baths.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("HalfBath")

#### BedroomAbvGr

$\text{Although BedroomAbvGr is a numerical feature, it is actually a categorical feature.}$

In [ ]:
train_data["BedroomAbvGr"] = train_data["BedroomAbvGr"].astype(str)
test_data["BedroomAbvGr"] = test_data["BedroomAbvGr"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["BedroomAbvGr"], name="BedroomAbvGr (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["BedroomAbvGr"], name="BedroomAbvGr (test data)")

$\text{Most of the houses have 3 bedrooms.}$<p>
$\text{There are not many houses with 0, 5 and 6 bedrooms.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("BedroomAbvGr")

#### KitchenAbvGr

$\text{Although KitchenAbvGr is a numerical feature, it is actually a categorical feature.}$

In [ ]:
train_data["KitchenAbvGr"] = train_data["KitchenAbvGr"].astype(str)
test_data["KitchenAbvGr"] = test_data["KitchenAbvGr"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["KitchenAbvGr"], name="KitchenAbvGr (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["KitchenAbvGr"], name="KitchenAbvGr (test data)")

$\text{Most of the houses have 1 kitchen.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("KitchenAbvGr")

#### KitchenQual

In [ ]:
plots.pie_and_bar_plot(data=train_data["KitchenQual"], name="KitchenQual (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["KitchenQual"], name="KitchenQual (test data)")

$\text{There is one missing value in test data.}$<p>
$\text{Based on the data description it might be Poor quality.}$<p>
$\text{Because Poor category is not present in train data, we will fill it with the second "worse" category - Fa (Fair).}$

In [ ]:
test_data["KitchenQual"].fillna("Fa", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("KitchenQual")

#### Functional

In [ ]:
plots.pie_and_bar_plot(data=train_data["Functional"], name="Functional (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Functional"], name="Functional (test data)")

$\text{There is one missing value in test data.}$<p>
$\text{Based on the data description it might be Sal (Salvage only).}$

In [ ]:
test_data["Functional"].fillna("Sal", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("Functional")

#### Fireplaces

$\text{Although Fireplaces is a numerical feature, due to the fact that there are only 4 unique values we can consider it as a categorical feature.}$

In [ ]:
train_data["Fireplaces"] = train_data["Fireplaces"].astype(str)
test_data["Fireplaces"] = test_data["Fireplaces"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["Fireplaces"], name="Fireplaces (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Fireplaces"], name="Fireplaces (test data)")

$\text{Most of the houses have no fireplaces.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("Fireplaces")

#### FireplaceQu

In [ ]:
plots.pie_and_bar_plot(data=train_data["FireplaceQu"], name="FireplaceQu (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["FireplaceQu"], name="FireplaceQu (test data)")

$\text{Based on data description, we will fill missing values with the category - No fireplace.}$

In [ ]:
train_data["FireplaceQu"].fillna("No fireplace", inplace=True)
test_data["FireplaceQu"].fillna("No fireplace", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("FireplaceQu")

#### GarageType

In [ ]:
plots.pie_and_bar_plot(data=train_data["GarageType"], name="GarageType (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["GarageType"], name="GarageType (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - No garage.}$

In [ ]:
train_data["GarageType"].fillna("No garage", inplace=True)
test_data["GarageType"].fillna("No garage", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("GarageType")

#### GarageFinish

In [ ]:
plots.pie_and_bar_plot(data=train_data["GarageFinish"], name="GarageFinish (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["GarageFinish"], name="GarageFinish (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - No garage.}$

In [ ]:
train_data["GarageFinish"].fillna("No garage", inplace=True)
test_data["GarageFinish"].fillna("No garage", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("GarageFinish")

#### GarageQual

In [ ]:
plots.pie_and_bar_plot(data=train_data["GarageQual"], name="GarageQual (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["GarageQual"], name="GarageQual (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - No garage.}$

In [ ]:
train_data["GarageQual"].fillna("No garage", inplace=True)
test_data["GarageQual"].fillna("No garage", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("GarageQual")

#### GarageCond

In [ ]:
plots.pie_and_bar_plot(data=train_data["GarageCond"], name="GarageCond (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["GarageCond"], name="GarageCond (test data)")

$\text{There are missing values in train and test data.}$<p>
$\text{Based on the data description, we will fill them with the category - No garage.}$

In [ ]:
train_data["GarageCond"].fillna("No garage", inplace=True)
test_data["GarageCond"].fillna("No garage", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("GarageCond")

#### PavedDrive

In [ ]:
plots.pie_and_bar_plot(data=train_data["PavedDrive"], name="PavedDrive (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["PavedDrive"], name="PavedDrive (test data)")

$\text{Most of the houses have paved driveway.}$<p>
$\text{There are not many houses with other types of driveway.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("PavedDrive")

#### PoolQC

In [ ]:
plots.pie_and_bar_plot(data=train_data["PoolQC"], name="PoolQC (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["PoolQC"], name="PoolQC (test data)")

$\text{Based on data description, we will fill missing values with the category - No pool.}$

In [ ]:
train_data["PoolQC"].fillna("No pool", inplace=True)
test_data["PoolQC"].fillna("No pool", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("PoolQC")

#### Fence

In [ ]:
plots.pie_and_bar_plot(data=train_data["Fence"], name="Fence (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["Fence"], name="Fence (test data)")

$\text{Based on data description, we will fill missing values with the category - No fence.}$

In [ ]:
train_data["Fence"].fillna("No fence", inplace=True)
test_data["Fence"].fillna("No fence", inplace=True)

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("Fence")

#### MiscFeature

In [ ]:
plots.pie_and_bar_plot(data=train_data["MiscFeature"], name="MiscFeature (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["MiscFeature"], name="MiscFeature (test data)")

$\text{Based on data description, we will fill missing values with the category - None.}$

In [ ]:
train_data["MiscFeature"].fillna("None", inplace=True)
test_data["MiscFeature"].fillna("None", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("MiscFeature")

#### YrSold

$\text{Although YrSold is a numerical feature, due to the fact that there are only 5 unique values we can consider it as a categorical feature.}$

In [ ]:
train_data["YrSold"] = train_data["YrSold"].astype(str)
test_data["YrSold"] = test_data["YrSold"].astype(str)

In [ ]:
plots.pie_and_bar_plot(data=train_data["YrSold"], name="YrSold (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["YrSold"], name="YrSold (test data)")

$\text{The distribution of YrSold is different in train and test data.}$

$\text{We can distinguish order of categories in this feature, so we will use ordinal encoding.}$

In [ ]:
ordinal.append("YrSold")

#### SaleType

In [ ]:
plots.pie_and_bar_plot(data=train_data["SaleType"], name="SaleType (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["SaleType"], name="SaleType (test data)")

$\text{There is one missing value in test data.}$<p>
$\text{We will fill it with the Other category.}$

In [ ]:
test_data["SaleType"].fillna("Other", inplace=True)

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("SaleType")

#### SaleCondition

In [ ]:
plots.pie_and_bar_plot(data=train_data["SaleCondition"], name="SaleCondition (train data)")

In [ ]:
plots.pie_and_bar_plot(data=test_data["SaleCondition"], name="SaleCondition (test data)")

$\text{Most of the houses are in Normal condition.}$<p>
$\text{There are not many houses with other conditions.}$

$\text{We can't distinguish order of categories in this feature, so we will use one-hot encoding.}$

In [ ]:
one_hot.append("SaleCondition")

### Continous features

#### LotFrontage

In [ ]:
plots.box_and_hist_plot(train=train_data["LotFrontage"], test=test_data["LotFrontage"], name="LotFrontage", bin_size=5)

$\text{Data has right skewed distribution.}$

In [ ]:
continous_features.append("LotFrontage")

#### LotArea

In [ ]:
plots.box_and_hist_plot(train=train_data["LotArea"], test=test_data["LotArea"], name="LotArea", bin_size=3000)

$\text{Data has right skewed distribution.}$

In [ ]:
continous_features.append("LotArea")

#### OverallQual

In [ ]:
plots.box_and_hist_plot(train=train_data["OverallQual"], test=test_data["OverallQual"], name="OverallQual", bin_size=1)

$\text{Most of the houses have average overall quality.}$

In [ ]:
continous_features.append("OverallQual")

#### OverallCond

In [ ]:
plots.box_and_hist_plot(train=train_data["OverallCond"], test=test_data["OverallCond"], name="OverallCond", bin_size=1)

$\text{Most of the houses have average overall condition.}$

In [ ]:
continous_features.append("OverallCond")

#### YearBuilt

In [ ]:
plots.box_and_hist_plot(train=train_data["YearBuilt"], test=test_data["YearBuilt"], name="YearBuilt", bin_size=2)

$\text{Most of the houses were built in 2000 and newer.}$

In [ ]:
continous_features.append("YearBuilt")

#### YearRemodAdd

In [ ]:
plots.box_and_hist_plot(train=train_data["YearRemodAdd"], test=test_data["YearRemodAdd"], name="YearRemodAdd", bin_size=2)

$\text{There are many houses which were remodelled in 1950 and after 2000.}$

In [ ]:
continous_features.append("YearRemodAdd")

#### MasVnrArea

In [ ]:
plots.box_and_hist_plot(train=train_data["MasVnrArea"], test=test_data["MasVnrArea"], name="MasVnrArea", bin_size=100)

$\text{Data has right skewed distribution.}$

In [ ]:
continous_features.append("MasVnrArea")

#### BsmtFinSF1

In [ ]:
plots.box_and_hist_plot(train=train_data["BsmtFinSF1"], test=test_data["BsmtFinSF1"], name="BsmtFinSF1", bin_size=100)

$\text{Data has right skewed distribution.}$

$\text{There are some missing values in test data.}$<p>
$\text{It probably means that there is no basement.}$<p>
$\text{We will fill missing values with 0.}$

In [ ]:
test_data["BsmtFinSF1"].fillna(0, inplace=True)

In [ ]:
continous_features.append("BsmtFinSF1")

#### BsmtFinSF2

In [ ]:
plots.box_and_hist_plot(train=train_data["BsmtFinSF2"], test=test_data["BsmtFinSF2"], name="BsmtFinSF2", bin_size=100)

$\text{Data has right skewed distribution.}$

$\text{Simmilarly to BsmtFinSF1, we will fill missing values with 0.}$

In [ ]:
test_data["BsmtFinSF2"].fillna(0, inplace=True)

In [ ]:
continous_features.append("BsmtFinSF2")

#### BsmtUnfSF

In [ ]:
plots.box_and_hist_plot(train=train_data["BsmtUnfSF"], test=test_data["BsmtUnfSF"], name="BsmtUnfSF", bin_size=50)

$\text{Data is right skewed distributed.}$

$\text{Simmilarly to BsmtFinSF1, we will fill missing values with 0.}$

In [ ]:
test_data["BsmtUnfSF"].fillna(0, inplace=True)

In [ ]:
continous_features.append("BsmtUnfSF")

#### TotalBsmtSF

In [ ]:
plots.box_and_hist_plot(train=train_data["TotalBsmtSF"], test=test_data["TotalBsmtSF"], name="TotalBsmtSF", bin_size=50)

$\text{Data is right skewed distributed.}$

$\text{Simmilarly to BsmtFinSF1, we will fill missing values with 0.}$

In [ ]:
test_data["TotalBsmtSF"].fillna(0, inplace=True)

In [ ]:
continous_features.append("TotalBsmtSF")

#### 1stFlrSF

In [ ]:
plots.box_and_hist_plot(train=train_data["1stFlrSF"], test=test_data["1stFlrSF"], name="1stFlrSF", bin_size=50)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("1stFlrSF")

#### 2ndFlrSF

In [ ]:
plots.box_and_hist_plot(train=train_data["2ndFlrSF"], test=test_data["2ndFlrSF"], name="2ndFlrSF", bin_size=50)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("2ndFlrSF")

#### LowQualFinSF

In [ ]:
plots.box_and_hist_plot(train=train_data["LowQualFinSF"], test=test_data["LowQualFinSF"], name="LowQualFinSF", bin_size=50)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("LowQualFinSF")

#### GrLivArea

In [ ]:
plots.box_and_hist_plot(train=train_data["GrLivArea"], test=test_data["GrLivArea"], name="GrLivArea", bin_size=50)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("GrLivArea")

#### TotRmsAbvGrd

In [ ]:
plots.box_and_hist_plot(train=train_data["TotRmsAbvGrd"], test=test_data["TotRmsAbvGrd"], name="TotRmsAbvGrd", bin_size=50)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("TotRmsAbvGrd")

#### GarageYrBlt

In [ ]:
plots.box_and_hist_plot(train=train_data["GarageYrBlt"], test=test_data["GarageYrBlt"], name="GarageYrBlt", bin_size=1)

$\text{Well it looks strange, because there is 1 observation with year 2207 in test data.}$

In [ ]:
test_data.loc[test_data["GarageYrBlt"] > 2010, "GarageYrBlt"]

$\text{That is probably a mistake, so we will replace it with the 2007.}$

In [ ]:
test_data.loc[test_data["GarageYrBlt"] > 2010, "GarageYrBlt"] = 2007

In [ ]:
plots.box_and_hist_plot(train=train_data["GarageYrBlt"], test=test_data["GarageYrBlt"], name="GarageYrBlt", bin_size=2)

$\text{Most of the houses have garage built in 2000 and newer.}$

$\text{It is also worth noting that there are some missing values in train and test data.}$<p>
$\text{It probably means that there is no garage.}$<p>
$\text{We will fill them with the 0.}$

In [ ]:
train_data["GarageYrBlt"].fillna(0, inplace=True)
test_data["GarageYrBlt"].fillna(0, inplace=True)

In [ ]:
continous_features.append("GarageYrBlt")

#### GarageCars

In [ ]:
plots.box_and_hist_plot(train=train_data["GarageCars"], test=test_data["GarageCars"], name="GarageCars", bin_size=1)

$\text{There are just several houses with 4 and 5 cars capacity.}$

$\text{Simmilarly to GarageYrBlt, we will fill missing values with 0.}$

In [ ]:
train_data["GarageCars"].fillna(0, inplace=True)
test_data["GarageCars"].fillna(0, inplace=True)

In [ ]:
continous_features.append("GarageCars")

#### GarageArea

In [ ]:
plots.box_and_hist_plot(train=train_data["GarageArea"], test=test_data["GarageArea"], name="GarageArea", bin_size=20)

$\text{Data is right skewed distributed with peak around 0 due to the fact that there are many houses with no garage.}$

$\text{Simmilarly to GarageYrBlt, we will fill missing values with 0.}$

In [ ]:
train_data["GarageArea"].fillna(0, inplace=True)
test_data["GarageArea"].fillna(0, inplace=True)

In [ ]:
continous_features.append("GarageArea")

#### WoodDeckSF

In [ ]:
plots.box_and_hist_plot(train=train_data["WoodDeckSF"], test=test_data["WoodDeckSF"], name="WoodDeckSF", bin_size=20)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("WoodDeckSF")

#### OpenPorchSF

In [ ]:
plots.box_and_hist_plot(train=train_data["OpenPorchSF"], test=test_data["OpenPorchSF"], name="OpenPorchSF", bin_size=10)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("OpenPorchSF")

#### EnclosedPorch

In [ ]:
plots.box_and_hist_plot(train=train_data["EnclosedPorch"], test=test_data["EnclosedPorch"], name="EnclosedPorch", bin_size=10)

$\text{Data is right skewed distributed.}$

In [ ]:
continous_features.append("EnclosedPorch")

#### 3SsnPorch

In [ ]:
plots.box_and_hist_plot(train=train_data["3SsnPorch"], test=test_data["3SsnPorch"], name="3SsnPorch", bin_size=10)

$\text{Data is right skewed distributed and most of the houses have no 3 season porch.}$

In [ ]:
continous_features.append("3SsnPorch")

#### ScreenPorch

In [ ]:
plots.box_and_hist_plot(train=train_data["ScreenPorch"], test=test_data["ScreenPorch"], name="ScreenPorch", bin_size=10)

$\text{Data is right skewed distributed with peak around 0 due to the fact that there are many houses with no screen porch.}$

In [ ]:
continous_features.append("ScreenPorch")

#### PoolArea

In [ ]:
plots.box_and_hist_plot(train=train_data["PoolArea"], test=test_data["PoolArea"], name="PoolArea", bin_size=10)

$\text{Data is right skewed distributed with peak around 0 due to the fact that there are many houses with no pool.}$

In [ ]:
continous_features.append("PoolArea")

#### MiscVal

In [ ]:
plots.box_and_hist_plot(train=train_data["MiscVal"], test=test_data["MiscVal"], name="MiscVal", bin_size=1000)

$\text{Data is right skewed distributed with peak around 0.}$

In [ ]:
continous_features.append("MiscVal")

#### MoSold

In [ ]:
plots.box_and_hist_plot(train=train_data["MoSold"], test=test_data["MoSold"], name="MoSold", bin_size=1)

$\text{Data is normally distributed.}$

In [ ]:
continous_features.append("MoSold")

## Target variable by categorical features

### MSSubClass

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="MSSubClass", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of MSSubClass.}$

### MSZoning

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="MSZoning", target="SalePrice")

$\text{For C (all) category, the SalePrice is lower than for other categories.}$

### Street

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Street", target="SalePrice")

$\text{Due to the fact that there are only 6 houses with Gravel street, we can't make any conclusion.}$

### Alley

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Alley", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Alley.}$

### LotShape

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="LotShape", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of LotShape.}$

### LandContour

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="LandContour", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of LandContour.}$

### Utilities

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Utilities", target="SalePrice")

$\text{Only 1 house has NoSeWa utilities, so we can't make any conclusion.}$

### LotConfig

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="LotConfig", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of LotConfig.}$

### LandSlope

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="LandSlope", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of LandSlope.}$

### Neighborhood

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Neighborhood", target="SalePrice")

$\text{There are some Neighborhoods with higher SalePrice than others.}$

### Condition1

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Condition1", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Condition1.}$

### Condition2

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Condition2", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Condition2.}$

### BldgType

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BldgType", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BldgType.}$

### HouseStyle

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="HouseStyle", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of HouseStyle.}$

### RoofStyle

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="RoofStyle", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of RoofStyle.}$

### RoofMatl

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="RoofMatl", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of RoofMatl.}$

### Exterior1st

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Exterior1st", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Exterior1st.}$

### Exterior2nd

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Exterior2nd", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Exterior2nd.}$

### MasVnrType

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="MasVnrType", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of MasVnrType.}$

### ExterQual

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="ExterQual", target="SalePrice")

$\text{It looks like Fa and TA categories have lower SalePrice than other categories.}$

### ExterCond

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="ExterCond", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of ExterCond.}$

### Foundation

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Foundation", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Foundation.}$

### BsmtQual

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtQual", target="SalePrice")

$\text{For TA, No basement and Fa categories, the SalePrice is lower than for other categories.}$

### BsmtCond

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtCond", target="SalePrice")

$\text{For Fa, No basement and Po categories, the SalePrice is lower than for Gd category.}$

### BsmtExposure

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtExposure", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BsmtExposure.}$

### BsmtFinType1

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtFinType1", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BsmtFinType1.}$

### BsmtFinType2

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtFinType2", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BsmtFinType2.}$

### Heating

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Heating", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Heating.}$

### HeatingQC

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="HeatingQC", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of HeatingQC.}$

### CentralAir

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="CentralAir", target="SalePrice")

$\text{For N category, the SalePrice is lower than for Y category.}$

### Electrical

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Electrical", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Electrical.}$

### BsmtFullBath

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtFullBath", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BsmtFullBath.}$

### BsmtHalfBath

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BsmtHalfBath", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BsmtHalfBath.}$

### FullBath

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="FullBath", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of FullBath.}$

### HalfBath

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="HalfBath", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of HalfBath.}$

### BedroomAbvGr

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="BedroomAbvGr", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of BedroomAbvGr.}$

### KitchenAbvGr

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="KitchenAbvGr", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of KitchenAbvGr.}$

### KitchenQual

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="KitchenQual", target="SalePrice")

$\text{For TA and Fa categories, the SalePrice is lower than for other categories.}$

### Functional

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Functional", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Functional.}$

### Fireplaces

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Fireplaces", target="SalePrice")

$\text{There is a positive correlation between number of fireplaces and SalePrice.}$

### FireplaceQu

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="FireplaceQu", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of FireplaceQu.}$

### GarageType

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="GarageType", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of GarageType.}$

### GarageFinish

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="GarageFinish", target="SalePrice")

$\text{For No garage category, the SalePrice is lower than for other categories.}$

### GarageQual

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="GarageQual", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of GarageQual.}$

### GarageCond

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="GarageCond", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of GarageCond.}$

### PavedDrive

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="PavedDrive", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of PavedDrive.}$

### PoolQC

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="PoolQC", target="SalePrice")

$\text{Most of the houses do not have pool, but for these that have (especially Ex), the SalePrice is higher.}$

### Fence

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="Fence", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of Fence.}$

### MiscFeature

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="MiscFeature", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of MiscFeature.}$

### YrSold

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="YrSold", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of YrSold.}$

### SaleType

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="SaleType", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of SaleType.}$

### SaleCondition

In [ ]:
plots.boxplot_by_categorical(data=train_data, categorical="SaleCondition", target="SalePrice")

$\text{There is no significant difference in SalePrice distribution between different categories of SaleCondition.}$

## Target variable by continous features

### LotFrontage

In [ ]:
plots.linear_regression_plot(data=train_data, feature="LotFrontage", target=target_variable)

### LotArea

In [ ]:
plots.linear_regression_plot(data=train_data, feature="LotArea", target=target_variable)

### OverallQual

In [ ]:
plots.linear_regression_plot(data=train_data, feature="OverallQual", target=target_variable)

### OverallCond

In [ ]:
plots.linear_regression_plot(data=train_data, feature="OverallCond", target=target_variable)

### YearBuilt

In [ ]:
plots.linear_regression_plot(data=train_data, feature="YearBuilt", target=target_variable)

### YearRemodAdd

In [ ]:
plots.linear_regression_plot(data=train_data, feature="YearRemodAdd", target=target_variable)

### MasVnrArea

In [ ]:
plots.linear_regression_plot(data=train_data, feature="MasVnrArea", target=target_variable)

### BsmtFinSF1

In [ ]:
plots.linear_regression_plot(data=train_data, feature="BsmtFinSF1", target=target_variable)

### BsmtFinSF2

In [ ]:
plots.linear_regression_plot(data=train_data, feature="BsmtFinSF2", target=target_variable)

### TotalBsmtSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="TotalBsmtSF", target=target_variable)

### 1stFlrSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="1stFlrSF", target=target_variable)

### 2ndFlrSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="2ndFlrSF", target=target_variable)

### LowQualFinSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="LowQualFinSF", target=target_variable)

### GrLivArea

In [ ]:
plots.linear_regression_plot(data=train_data, feature="GrLivArea", target=target_variable)

### TotRmsAbvGrd

In [ ]:
plots.linear_regression_plot(data=train_data, feature="TotRmsAbvGrd", target=target_variable)

### GarageYrBlt

In [ ]:
plots.linear_regression_plot(data=train_data, feature="GarageYrBlt", target=target_variable)

### GarageCars

In [ ]:
plots.linear_regression_plot(data=train_data, feature="GarageCars", target=target_variable)

### GarageArea

In [ ]:
plots.linear_regression_plot(data=train_data, feature="GarageArea", target=target_variable)

### WoodDeckSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="WoodDeckSF", target=target_variable)

### OpenPorchSF

In [ ]:
plots.linear_regression_plot(data=train_data, feature="OpenPorchSF", target=target_variable)

### EnclosedPorch

In [ ]:
plots.linear_regression_plot(data=train_data, feature="EnclosedPorch", target=target_variable)

### 3SsnPorch

In [ ]:
plots.linear_regression_plot(data=train_data, feature="3SsnPorch", target=target_variable)

### ScreenPorch

In [ ]:
plots.linear_regression_plot(data=train_data, feature="ScreenPorch", target=target_variable)

### PoolArea

In [ ]:
plots.linear_regression_plot(data=train_data, feature="PoolArea", target=target_variable)

### MiscVal

In [ ]:
plots.linear_regression_plot(data=train_data, feature="MiscVal", target=target_variable)

### MoSold

In [ ]:
plots.linear_regression_plot(data=train_data, feature="MoSold", target=target_variable)

$\text{Let's take a look how missing values are distributed in the data after imputing some of categorical features.}$

In [ ]:
missing_data_train = pd.DataFrame(train_data.isna().sum()/train_data.shape[0], columns=['Missing Percentage']).sort_values(by='Missing Percentage',ascending=False)
missing_data_train = missing_data_train.loc[missing_data_train['Missing Percentage']>0, :]
missing_columns = list(missing_data_train.index)
plots.barplot_missing_values(data=missing_data_train, features_names=missing_columns, name="missing values (train)")

In [ ]:
missing_data_test = pd.DataFrame(test_data.isna().sum()/test_data.shape[0], columns=['Missing Percentage']).sort_values(by='Missing Percentage',ascending=False)
missing_data_test = missing_data_test.loc[missing_data_test['Missing Percentage']>0, :]
missing_columns = list(missing_data_test.index)
plots.barplot_missing_values(data=missing_data_test, features_names=missing_columns, name="missing values (train)")

$\text{As we can see, there are still some missing values in the data.}$

$\text{There is one missing value for Electrical feature in train data.}$<p>
$\text{We will fill it with the most frequent category - "Sbrkr".}$

In [ ]:
train_data["Electrical"].fillna(train_data["Electrical"].mode()[0], inplace=True)

$\text{For MasVnrArea feature, missing values most likely means that there is no masonry veneer.}$<p>
$\text{We will fill them with the 0.}$

In [ ]:
train_data["MasVnrArea"].fillna(0, inplace=True)
test_data["MasVnrArea"].fillna(0, inplace=True)

$\text{For LotFrontage we will fill missing values with median LotFrontage in the same neighborhood.}$<p>
$\text{Since the area of each street connected to the house property most likely have a similar area to other houses in its neighborhood.}$

In [ ]:
train_data["LotFrontage"] = train_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
test_data["LotFrontage"] = test_data.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

$\text{We will check if there are any missing values left.}$

In [ ]:
missing_data_train = pd.DataFrame(train_data.isna().sum()/train_data.shape[0], columns=['Missing Percentage']).sort_values(by='Missing Percentage',ascending=False)
missing_data_train = missing_data_train.loc[missing_data_train['Missing Percentage']>0, :]
missing_data_train

In [ ]:
missing_data_test = pd.DataFrame(test_data.isna().sum()/test_data.shape[0], columns=['Missing Percentage']).sort_values(by='Missing Percentage',ascending=False)
missing_data_test = missing_data_test.loc[missing_data_test['Missing Percentage']>0, :]
missing_data_test

$\text{Since there are no missing values left, we can proceed with encoding categorical features.}$

## Encoding categorical features

$\text{We created two lists with one-hot and ordinal features.}$<p>
$\text{We will use one-hot encoding for one-hot features and ordinal encoding for ordinal features.}$

In [ ]:
for feature in one_hot:
    unique_values = np.sort(train_data[feature].unique())
    unknown_test_indices = test_data.loc[~test_data[feature].isin(unique_values)].index.tolist()
    one_hot_encoder = OneHotEncoder(drop="first", categories=[unique_values], handle_unknown='ignore', sparse_output=False)
    encoded_cols_drop_first = [f"{feature}_{val}" for val in np.delete(unique_values, 0)]
    one_hot_encoder.fit(train_data[feature].values.reshape(-1, 1))
    train_transformed = one_hot_encoder.transform(train_data[feature].values.reshape(-1, 1))
    test_transformed = one_hot_encoder.transform(test_data[feature].values.reshape(-1, 1))
    train_transformed = pd.DataFrame(train_transformed, columns=encoded_cols_drop_first)
    test_transformed = pd.DataFrame(test_transformed, columns=encoded_cols_drop_first)
    test_transformed.loc[unknown_test_indices, :] = -1
    train_data = pd.concat([train_data, train_transformed], axis=1).drop(feature, axis=1)
    test_data = pd.concat([test_data, test_transformed], axis=1).drop(feature, axis=1)

In [ ]:
for feature in ordinal:
    unique_values = np.sort(train_data[feature].unique())
    ordinal_encoder = OrdinalEncoder(categories=[unique_values], handle_unknown='use_encoded_value', unknown_value=-1)
    ordinal_encoder.fit(train_data[feature].values.reshape(-1, 1))
    train_data[feature] = ordinal_encoder.transform(train_data[feature].values.reshape(-1, 1))
    test_data[feature] = ordinal_encoder.transform(test_data[feature].values.reshape(-1, 1))

## Univariate Outliers detection

$\text{We will use several techniques to detect outliers by univariate methods.}$

### Standard deviation method

In [ ]:
class STD_Method():
    def __init__(self):
        self.fit_used = False

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        return np.array(data)
    
    def check_for_object_columns(self, data):
        data = pd.DataFrame(data)
        if data.select_dtypes(include=np.number).shape[1] != data.shape[1]:
            raise TypeError('Your data contains object or string columns. Numeric data is obligated.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return np.array(data)
    
    def fit(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.std_ = np.std(data)
        self.mean_ = np.mean(data)
        self.fit_used = True

    def find_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.indices_of_outliers_ = np.where((data < self.mean_-3*self.std_) | (data > self.mean_+3*self.std_))[0]
        return self.indices_of_outliers_
    
    def remove_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.delete(data, self.indices_of_outliers_)
    
    def check_fit(self, fit_used):
        if fit_used == False:
            raise AttributeError('STD_Method has to be fitted first.')

In [ ]:
STD_transformer = STD_Method()
outliers_dict = {indice: 0 for indice in range(0, train_data.shape[0])}
for feature in continous_features:
    STD_transformer.fit(data=train_data[feature])
    outliers_indices = STD_transformer.find_outliers(data=train_data[feature])
    for outlier_idx in outliers_indices:
        outliers_dict[outlier_idx] += 1
outliers_dict = dict(sorted(outliers_dict.items(), key=lambda item: item[1], reverse=True))
for idx, (key, value) in enumerate(outliers_dict.items()):
    print(f"Sample {key} was detected as outlier {value} times (for {round(value/train_data.shape[0]*100, 4)}% of features) based on STD Method.")
    if(idx == 5):
        break

$\text{As we can see the sample 1298 was detected as outlier in more than half of our features.}$<p>
$\text{We will keep that in mind and try other techniques}$

### Z-Score

In [ ]:
class Z_Score():
    def __init__(self, threshold=3):
        self.fit_used = False
        self.threshold = threshold

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        return np.array(data)
    
    def check_for_object_columns(self, data):
        data = pd.DataFrame(data)
        if data.select_dtypes(include=np.number).shape[1] != data.shape[1]:
            raise TypeError('Your data contains object or string columns. Numeric data is obligated.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return np.array(data)
    
    def fit(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.std_ = np.std(data)
        self.mean_ = np.mean(data)
        self.fit_used = True

    def find_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.indices_of_outliers_ = np.where(np.abs((data-self.mean_)/self.std_)>self.threshold)[0]
        return self.indices_of_outliers_
    
    def remove_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.delete(data, self.indices_of_outliers_)
    
    def check_fit(self, fit_used):
        if fit_used == False:
            raise AttributeError('Z_Score has to be fitted first.')

$\text{We will iterate through every continous features and check whether there are any indices which were detected as outliers for more than half of our features.}$

In [ ]:
Z_score_transformer = Z_Score(threshold=3)
outliers_dict = {indice: 0 for indice in range(0, train_data.shape[0])}
for feature in continous_features:
    Z_score_transformer.fit(data=train_data[feature])
    outliers_indices = Z_score_transformer.find_outliers(data=train_data[feature])
    for outlier_idx in outliers_indices:
        outliers_dict[outlier_idx] += 1
outliers_dict = dict(sorted(outliers_dict.items(), key=lambda item: item[1], reverse=True))
for idx, (key, value) in enumerate(outliers_dict.items()):
    print(f"Sample {key} was detected as outlier {value} times (for {round(value/train_data.shape[0]*100, 4)}% of features) based on Z-Score.")
    if(idx == 5):
        break

$\text{Sample 1298 was detected once again as outlier.}$

### Robust Z-Score

In [ ]:
class Robust_Z_Score():
    def __init__(self, const=0.6745, threshold=3):
        self.fit_used = False
        self.const_ = const
        self.threshold = threshold

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        return np.array(data)
    
    def check_for_object_columns(self, data):
        data = pd.DataFrame(data)
        if data.select_dtypes(include=np.number).shape[1] != data.shape[1]:
            raise TypeError('Your data contains object or string columns. Numeric data is obligated.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return np.array(data)
    
    def fit(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.median_ = np.median(data)
        self.MAD_ = np.median(np.abs(data-np.median(data)))
        self.fit_used = True

    def find_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.indices_of_outliers_ = np.where(np.abs(self.const_*(data-self.median_)/self.MAD_)>self.threshold)[0]
        return self.indices_of_outliers_
    
    def remove_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.delete(data, self.indices_of_outliers_)
    
    def check_fit(self, fit_used):
        if fit_used == False:
            raise AttributeError('Robust_Z_Score has to be fitted first.')

In [ ]:
Robust_Z_score_transformer = Robust_Z_Score(const=0.6745, threshold=3)
outliers_dict = {indice: 0 for indice in range(0, train_data.shape[0])}
for feature in continous_features:
    Robust_Z_score_transformer.fit(data=train_data[feature])
    outliers_indices = Robust_Z_score_transformer.find_outliers(data=train_data[feature])
    for outlier_idx in outliers_indices:
        outliers_dict[outlier_idx] += 1
outliers_dict = dict(sorted(outliers_dict.items(), key=lambda item: item[1], reverse=True))
for idx, (key, value) in enumerate(outliers_dict.items()):
    print(f"Sample {key} was detected as outlier {value} times (for {round(value/train_data.shape[0]*100, 4)}% of features) based on Robust Z-Score.")
    if(idx == 5):
        break

$\text{Sample 1298 was detected once again as outlier.}$

In [ ]:
class Inter_Quantile_Range():
    def __init__(self):
        self.fit_used = False

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        return np.array(data)
    
    def check_for_object_columns(self, data):
        data = pd.DataFrame(data)
        if data.select_dtypes(include=np.number).shape[1] != data.shape[1]:
            raise TypeError('Your data contains object or string columns. Numeric data is obligated.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return np.array(data)
    
    def fit(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.Q1 = np.quantile(data, q=0.25)
        self.Q3 = np.quantile(data, q=0.75)
        self.IQR = self.Q3-self.Q1
        self.fit_used = True

    def find_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        self.indices_of_outliers_ = np.where((data < self.Q1-1.5*self.IQR) | (data > self.Q3+1.5*self.IQR))[0]
        return self.indices_of_outliers_
    
    def remove_outliers(self, data):
        self.check_fit(fit_used=self.fit_used)
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.delete(data, self.indices_of_outliers_)
    
    def check_fit(self, fit_used):
        if fit_used == False:
            raise AttributeError('Inter_Quantile_Range has to be fitted first.')

In [ ]:
IQR_transformer = Inter_Quantile_Range()
outliers_dict = {indice: 0 for indice in range(0, train_data.shape[0])}
for feature in continous_features:
    IQR_transformer.fit(data=train_data[feature])
    outliers_indices = IQR_transformer.find_outliers(data=train_data[feature])
    for outlier_idx in outliers_indices:
        outliers_dict[outlier_idx] += 1
outliers_dict = dict(sorted(outliers_dict.items(), key=lambda item: item[1], reverse=True))
for idx, (key, value) in enumerate(outliers_dict.items()):
    print(f"Sample {key} was detected as outlier {value} times (for {round(value/train_data.shape[0]*100, 4)}% of features) based on IQR.")
    if(idx == 5):
        break

$\text{Sample 1298 was detected once again as outlier.}$

## Multivariate Outliers detection

$\text{Univariate methods are good when it comes to detection of "strange" samples for signle features.}$<p>
$\text{However due to the fact that data contains houndreds of variables it is better to rely on multivariate methods.}$<p>
$\text{We will compare several multivariate methods and see which one is the best.}$<p>
$\text{To do that, firstly we will calculate base score.}$

### Base score

In [ ]:
categorical_features = ordinal+[feature for feature in train_data.columns if train_data[feature].nunique() < 10]

In [ ]:
lgbm_params = {
    "objective": "regression",
    "metric": "mape",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 17,
    "n_jobs": 5
}
cv = KFold(n_splits=10, shuffle=True, random_state=17)
X = train_data.drop(target_variable, axis=1)
y = train_data[target_variable]
scores = []
model = LGBMRegressor(**lgbm_params)
for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
    y_pred = model.predict(X_valid)
    mape = mean_absolute_percentage_error(y_valid, y_pred)
    scores.append(mape)
print(f"Base score: {np.mean(scores)}")

### Mahalanobis distance

In [ ]:
def objective(trial, X, y):
    mahalanobis_param = {
        "alpha": trial.suggest_float("alpha", 0.001, 0.2),
    }
    scores = []
    model = LGBMRegressor(**lgbm_params)
    for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        empirical_cov = EmpiricalCovariance().fit(X_train)
        mahal_emp_cov = empirical_cov.mahalanobis(X_train)**0.5
        mahal_emp_cov = np.sort(mahal_emp_cov)
        threshold = np.sqrt(chi2.ppf(1-mahalanobis_param["alpha"], df=X_train.shape[1]))
        X_train_no_outliers = X_train[mahal_emp_cov < threshold]
        y_train_no_outliers = y_train[mahal_emp_cov < threshold]
        model.fit(X_train_no_outliers, y_train_no_outliers, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
        y_pred = model.predict(X_valid)
        mape = mean_absolute_percentage_error(y_valid, y_pred)
        scores.append(mape)
    return np.mean(scores)
sampler = optuna.samplers.TPESampler(seed=17)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(lambda trial: objective(trial, X, y), n_trials=100)
best_params = study.best_params
best_alpha = best_params["alpha"]

$\text{We obtained worse results after removal of outliers based on Mahalanobis distance compared to base score.}$

In [ ]:
empirical_cov = EmpiricalCovariance().fit(X)
mahal_emp_cov = empirical_cov.mahalanobis(X)**0.5
mahal_emp_cov = np.sort(mahal_emp_cov)
threshold = np.sqrt(chi2.ppf(1-best_alpha, df=X.shape[1]))
X_no_outliers = X[mahal_emp_cov <= threshold]
y_no_outliers = y[mahal_emp_cov <= threshold]
mahal_indices_of_outliers = X.index[mahal_emp_cov > threshold].tolist()
print(f"Number of outliers: {X.shape[0] - X_no_outliers.shape[0]}")
print("Ratio of outliers:", (X.shape[0] - X_no_outliers.shape[0]) / X.shape[0])

$\text{Well it looks like Mahalanobis method returned to many outliers.}$<p>
$\text{It is probably due to the fact that it works best for continous features and prefer them to be normally distributed.}$

### Isolation Forest

In [ ]:
def objective(trial, X, y):
    isolation_forest_param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "contamination": trial.suggest_float("contamination", 0.005, 0.05),
        "max_samples": trial.suggest_float("max_samples", 0.2, 0.4),
        "bootstrap": True,
        "n_jobs": 3,
        "random_state": 17,
        "verbose": 0
    }
    isolation_forest = IsolationForest(**isolation_forest_param)
    scores = []
    model = LGBMRegressor(**lgbm_params)
    for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        isolation_forest.fit(X_train)
        X_train_no_outliers = X_train[isolation_forest.predict(X_train) == 1]
        y_train_no_outliers = y_train[isolation_forest.predict(X_train) == 1]
        model.fit(X_train_no_outliers, y_train_no_outliers, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
        y_pred = model.predict(X_valid)
        mape = mean_absolute_percentage_error(y_valid, y_pred)
        scores.append(mape)
    return np.mean(scores)
sampler = optuna.samplers.TPESampler(seed=17)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=100)
isolation_forest_best_params = study.best_params

$\text{We obtained better results after removal of outliers based on Isolation Forest compared to base score.}$

In [ ]:
isolation_forest = IsolationForest(**isolation_forest_best_params)
isolation_forest.fit(X)
X_no_outliers = X[isolation_forest.predict(X) == 1]
y_no_outliers = y[isolation_forest.predict(X) == 1]
isolation_forest_indices_of_outliers = X.index[isolation_forest.predict(X) == -1].tolist()
print("Number of outliers:", X.shape[0] - X_no_outliers.shape[0])
print("Ratio of outliers:", (X.shape[0] - X_no_outliers.shape[0]) / X.shape[0])

### Local Outlier Factor

$\text{Before performing outliers detection using LOF we should scale the data.}$

In [ ]:
X_lof = X.copy()
scaler = StandardScaler()
X_lof[continous_features] = scaler.fit_transform(X_lof[continous_features])

In [ ]:
def objective(trial, X, y):
    lof_param = {
        "n_neighbors": trial.suggest_int("n_neighbors", 5, 40),
        "contamination": trial.suggest_float("contamination", 0.005, 0.05),
        "n_jobs": 3,
    }
    lof = LocalOutlierFactor(**lof_param)
    scores = []
    model = LGBMRegressor(**lgbm_params)
    for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        X_train_no_outliers = X_train[lof.fit_predict(X_train) == 1]
        y_train_no_outliers = y_train[lof.fit_predict(X_train) == 1]
        model.fit(X_train_no_outliers, y_train_no_outliers, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
        y_pred = model.predict(X_valid)
        mape = mean_absolute_percentage_error(y_valid, y_pred)
        scores.append(mape)
    return np.mean(scores)
sampler = optuna.samplers.TPESampler(seed=17)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(lambda trial: objective(trial, X_lof, y), n_trials=100)
lof_best_params = study.best_params

In [ ]:
lof = LocalOutlierFactor(**lof_best_params)
X_no_outliers = X[lof.fit_predict(X_lof) == 1]
y_no_outliers = y[lof.fit_predict(X_lof) == 1]
lof_indices_of_outliers = X.index[lof.fit_predict(X_lof) == -1].tolist()
print("Number of outliers:", X.shape[0] - X_no_outliers.shape[0])
print("Ratio of outliers:", (X.shape[0] - X_no_outliers.shape[0]) / X.shape[0])

$\text{We obtained a bit better results after removal of outliers based on LOF compared to base score.}$

In [ ]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, n_jobs=-1, random_state=17)
tsne_matrix = tsne.fit_transform(train_data)
plots.tsne_plot(tsne_matrix, isolation_forest_indices_of_outliers)

In [ ]:
print("Indices of outliers based on Isolation Forest: {}".format(isolation_forest_indices_of_outliers))

$\text{Outliers chosen by RandomForest looks pretty reasonable based on TSNE visualization.}$<p>
$\text{Moreover we obtained the best results after removal of outliers based on RandomForest compared to base score.}$<p>
$\text{Also the indices of outliers chosen by RandomForest are the most simmilar to the indices of outliers chosen by univariate methods.}$

In [ ]:
train_data.drop(isolation_forest_indices_of_outliers, axis=0, inplace=True)
train_data.reset_index(drop=True, inplace=True)

## Fixing skewness

$\text{Skewnness of the data is not a problem for tree based models, but it is for linear models.}$<p>
$\text{We will first calculate skewness of the continous features.}$

In [ ]:
class Skewness:
    def __init__(self):
        pass

    def check_data(self, data):
        if not isinstance(data, pd.DataFrame) and not isinstance(data, pd.Series) and not isinstance(data, np.ndarray) and not torch.is_tensor(data):
            raise TypeError('Wrong type of data. It should be pandas DataFrame, pandas Series, numpy array or torch tensor.')
        data = np.array(data)
        if(data.ndim == 2):
            data = data.squeeze()
        return data
    
    def check_for_object_columns(self, data):
        data = pd.DataFrame(data)
        if data.select_dtypes(include=np.number).shape[1] != data.shape[1]:
            raise TypeError('Your data contains object or string columns. Numeric data is obligated.')
        return np.array(data)

    def calculate_skewness(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.sum((data-np.mean(data))**3)/((data.shape[0]-1)*np.std(data)**3)

    def logarithmic_transformation(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.log(data)
    
    def square_root_transformation(self, data):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        return np.sqrt(data)
    
    def box_cox_transformation(self, data, reg_lambda=0):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        if(reg_lambda==0):
            return self.logarithmic_transformation(data=data)
        else:
            return (data**reg_lambda-1)/reg_lambda
    
    def inverse_transform_box_cox(self, data, reg_lambda=0):
        data = self.check_data(data=data)
        data = self.check_for_object_columns(data=data)
        if(reg_lambda==0):
            return np.exp(data)
        else:
            return (data*reg_lambda+1)**(1/reg_lambda)

In [ ]:
skew = Skewness()
for feature in continous_features:
    print("Skewness of {} variable is: {}".format(feature, skew.calculate_skewness(data=train_data[feature])))

$\text{Before we will transform features to remove skewness, we will calculate current base score with cross validation.}$

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=17)
X = train_data.drop(target_variable, axis=1)
y = train_data[target_variable]
scores = []
for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
    y_pred = model.predict(X_valid)
    mape = mean_absolute_percentage_error(y_valid, y_pred)
    scores.append(mape)
print(f"Base score: {np.mean(scores)}")

$\text{Let's start with transformation of target variable.}$

In [ ]:
_, optimal_lambda = boxcox(train_data[target_variable])
train_data_temp = train_data.copy()
train_data_temp[target_variable] = skew.box_cox_transformation(data=train_data_temp[target_variable], reg_lambda=optimal_lambda)

$\text{Now we can visualize our target variable and see that it is more normally distributed.}$

In [ ]:
plots.histogram_and_box_plot(data=train_data_temp[target_variable], name=target_variable, with_annotation=True)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=17)
X = train_data_temp.drop(target_variable, axis=1)
y = train_data_temp[target_variable]
scores = []
model = LGBMRegressor(**lgbm_params)
for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
    y_pred = model.predict(X_valid)
    mape = mean_absolute_percentage_error(y_valid, y_pred)
    scores.append(mape)
print(f"Base score: {np.mean(scores)}")

$\text{Wow it looks like we were able to decrease mean absolute percentage error more than two times.}$

$\text{Therefore, we will convert target variable in original train data.}$<p>
$\text{We have to remember to invert transform test predictions after whole modelling stage.}$

In [ ]:
_, optimal_lambda_target_variable = boxcox(train_data[target_variable])
train_data[target_variable] = skew.box_cox_transformation(data=train_data[target_variable], reg_lambda=optimal_lambda_target_variable)

## Feature Selection

$\text{We will use several techniques to select the most important features.}$<p>
$\text{Firslty, we will check the importance of features based on split and gain.}$

In [ ]:
def cross_validation_feature_importance(X, y, metric, n_splits=5, random_state=17, early_stopping_rounds=100):
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    gain_importances, split_importances, shap_importances = [], [], []
    for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=False)
        lgb_valid = lgb.Dataset(X_valid, y_valid, free_raw_data=False)
        lgb_model = lgb.train(lgbm_params, lgb_train, valid_sets=[lgb_valid], valid_names=['valid'], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
        shap_values = shap.TreeExplainer(lgb_model).shap_values(X_valid)
        shap_importance = np.abs(shap_values).mean(axis=0)
        normalized_shap_importance = shap_importance/np.sum(shap_importance)
        gain_importance = lgb_model.feature_importance(importance_type = 'gain')
        normalized_gain_importance = np.array(gain_importance)/np.sum(gain_importance)
        split_importance = lgb_model.feature_importance(importance_type = 'split')
        normalized_split_importance = np.array(split_importance)/np.sum(split_importance)
        gain_importances.append(normalized_gain_importance)
        split_importances.append(normalized_split_importance)
        shap_importances.append(normalized_shap_importance)
    results = {'feature': X.columns.tolist(), 'gain_std': np.std(gain_importances, axis=0), 'gain_mean': np.mean(gain_importances, axis=0), 'split_std': np.std(split_importances, axis=0), 'split_mean': np.mean(split_importances, axis=0), 'shap_std': np.std(shap_importances, axis=0), 'shap_mean': np.mean(shap_importances, axis=0)}
    results_df = pd.DataFrame(results)
    return results_df
metric = "mape"
results = cross_validation_feature_importance(X, y, metric, n_splits=10, random_state=17, early_stopping_rounds=50)

In [ ]:
results["all_methods_mean"] = results[["gain_mean", "split_mean", "shap_mean"]].mean(axis=1)
results["all_methods_std"] = results[["gain_std", "split_std", "shap_std"]].mean(axis=1)
for importance_type in ["gain", "split", "shap", "all_methods"]:
    plots.plot_feature_importances(results, importance_type=importance_type)

In [ ]:
results["ranking"] = results["all_methods_mean"].rank(ascending=False).astype(int)
results

In [ ]:
class Gradient_Feature_selection():
    def __init__(self, random_state):
        self.random_state = random_state
        np.random.seed(self.random_state)
        random.seed(self.random_state)

    def check_data(self, X, y):
        if isinstance(X, pd.DataFrame) and isinstance(y, pd.Series):
            return X, y
        else:
            raise ValueError("X and y should be pandas dataframe and series respectively")
    
    def check_metric(self, metric):
        metrics = {"accuracy": [lambda y, y_pred: accuracy_score(y, y_pred), "preds"],
                    "roc_auc": [lambda y, y_pred: roc_auc_score(y, y_pred), "probs"],
                    "f1": [lambda y, y_pred: f1_score(y, y_pred, average="weighted"), "preds"],
                    "auc_mu": [lambda y, y_pred: roc_auc_score(y, y_pred, multi_class='ovr'), "probs"],
                    "mse": [lambda y, y_pred: mean_squared_error(y, y_pred), "preds"],
                    "rmse": [lambda y, y_pred: mean_squared_error(y, y_pred)**0.5, "preds"],
                    "mae": [lambda y, y_pred: mean_absolute_error(y, y_pred), "preds"],
                    "mape": [lambda y, y_pred: mean_absolute_percentage_error(y, y_pred), "preds"]}
        if metric in metrics:
            return metrics[metric][0], metrics[metric][1]
        else:
            raise ValueError("Invalid metric. Please choose from the following: accuracy, roc_auc, f1, auc_mu, mse, rmse, mae")
    
    def fit(self, algorithm, X, y, metric, n_splits=5, learning_rate=1e-2, n_iter=100, objective="maximize"):
        X, y = self.check_data(X, y)
        self.eval_metric, self.metric_type = self.check_metric(metric)
        self.algorithm = algorithm
        self.n_splits = n_splits
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.objective = objective
        self.feature_importance_, self.selected_features_, self.best_score_ = self.perform_feature_selection(X, y)
        return self
    
    def fit_transform(self, algorithm, X, y, metric, n_splits=5, learning_rate=1e-2, n_iter=100, objective="maximize"):
        self.fit(algorithm, X, y, metric, n_splits, learning_rate, n_iter, objective)
        return X[self.selected_features_]
    
    def transform(self, X):
        return X[self.selected_features_]

    def perform_feature_selection(self, X, y):
        self.base_score = self.cross_val_score(X, y)
        best_score = self.base_score
        best_subset = X.columns.tolist()
        used_subset = set()
        feature_weights = {feature: 1/X.shape[1] for feature in X.columns}
        number_of_possible_subsets = 2**X.shape[1]
        for iter in range(self.n_iter):
            if(iter==number_of_possible_subsets):
                break
            while True:
                feature_subset = np.random.choice(X.columns, size=np.random.randint(1, X.shape[1]), replace=False, p=[feature_weights[feature] for feature in X.columns])
                if tuple(feature_subset) not in used_subset:
                    break
            X_subset = X[feature_subset]
            subset_score = self.cross_val_score(X_subset, y)
            loss = self.base_score - subset_score
            if(self.objective=="maximize"):
                if(subset_score > best_score):
                    best_score = subset_score
                    best_subset = feature_subset.tolist()
                feature_weights.update({feature: np.min([np.max([feature_weights[feature]-self.learning_rate*loss, 0]), 1]) if feature in feature_subset else feature_weights[feature] for feature in X.columns})
            else:
                if(subset_score < best_score):
                    best_score = subset_score
                    best_subset = feature_subset.tolist()
                feature_weights.update({feature: np.min([np.max([feature_weights[feature]+self.learning_rate*loss, 0]), 1]) if feature in feature_subset else feature_weights[feature] for feature in X.columns})
            feature_weights = {feature: feature_weights[feature]/sum(feature_weights.values()) for feature in feature_weights}
            if(iter%50==0):
                print(f"Iteration: {iter}, Best score: {best_score}; best subset: {best_subset}")
            used_subset.add(tuple(best_subset))
        return feature_weights, best_subset, best_score
    
    def cross_val_score(self, X, y):
        cv = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        cross_validation_scores = []
        for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]
            self.algorithm.fit(X_train, y_train)
            if(self.metric_type == "preds"):
                y_pred = self.algorithm.predict(X_valid)
            else:
                y_pred = self.algorithm.predict_proba(X_valid)[:,-1]
            cross_validation_scores.append(self.eval_metric(y_valid, y_pred))
        return np.mean(cross_validation_scores)

In [ ]:
gradient_selector = Gradient_Feature_selection(random_state=17)
gradient_selector.fit(LGBMRegressor(**lgbm_params), X, y, "mape", n_splits=5, learning_rate=1e-2, n_iter=500, objective="minimize")

In [ ]:
results = pd.DataFrame(gradient_selector.feature_importance_, index=["gradient_importance"]).T
plots.simple_feature_importance(results, importance_type="gradient_importance")

In [ ]:
print("Selected features: {}".format(gradient_selector.selected_features_))

In [ ]:
print("Base score after training with all features: {}".format(gradient_selector.base_score))
print("Best score after training with selected features: {}".format(gradient_selector.best_score_))

$\text{We will use only features selected by gradient feature selection.}$

In [ ]:
train_data = train_data[gradient_selector.selected_features_+[target_variable]]
test_data = test_data[gradient_selector.selected_features_]

## Hyperparameter tuning

In [ ]:
X = train_data.drop(target_variable, axis=1)
y = train_data[target_variable]

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=17)
def objective(trial, X, y):
    param = {
        "objective": "regression",
        "metric": "mape",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "random_state": 17,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.05),
        "n_estimators": trial.suggest_int("n_estimators", 80, 600),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.005, 0.015),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50)
    }
    scores = []
    model = LGBMRegressor(**param)
    for iter, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_names=['valid'], eval_metric="mape", callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=0)])
        y_pred = model.predict(X_valid)
        mape = mean_absolute_percentage_error(y_valid, y_pred)
        scores.append(mape)
    return np.mean(scores)
sampler = optuna.samplers.TPESampler(seed=17)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(lambda trial: objective(trial, X, y), n_trials=100)
best_params = study.best_params
print(best_params)

In [ ]:
best_params["objective"] = "regression"
best_params["metric"] = "mape"
best_params["verbosity"] = -1
best_params["boosting_type"] = "gbdt"
best_params["random_state"] = 17

# Modelling

In [ ]:
final_model = LGBMRegressor(**best_params)
final_model.fit(X, y)
y_train_pred = final_model.predict(X)

In [ ]:
y_train_pred_inversed = skew.inverse_transform_box_cox(data=y_train_pred, reg_lambda=optimal_lambda_target_variable)
y_inversed = skew.inverse_transform_box_cox(data=y, reg_lambda=optimal_lambda_target_variable)

In [ ]:
plots.compare_predictions_with_real_values(y_inversed, y_train_pred_inversed, "MAPE")

In [ ]:
submission = pd.read_csv("input/sample_submission.csv")
y_test_pred = final_model.predict(test_data)
y_test_pred_inversed = skew.inverse_transform_box_cox(data=y_test_pred, reg_lambda=optimal_lambda_target_variable)
submission[target_variable] = y_test_pred_inversed
submission.to_csv("input/submission.csv", index=False)
submission.head()

In [ ]:
plots.histogram_and_box_plot(data=y_test_pred_inversed, name=target_variable, with_annotation=True)

$\text{Distribution of predicted values looks quite simmilar to our training data target distribution - that's a good sign!.}$